In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
from sklearn.model_selection import train_test_split, cross_val_score, RandomizedSearchCV, GridSearchCV, StratifiedKFold, KFold
from sklearn.feature_selection import RFE
from sklearn.preprocessing import OrdinalEncoder, StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression, Ridge, Lasso, ElasticNet, RidgeCV
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier, HistGradientBoostingRegressor, RandomForestRegressor
import xgboost as xgb
from sklearn.cluster import KMeans, DBSCAN
from sklearn.metrics import accuracy_score, confusion_matrix, mean_absolute_error, r2_score, classification_report, roc_auc_score, RocCurveDisplay, f1_score, recall_score
from sklearn.linear_model import LinearRegression
from sklearn.metrics import classification_report
from scipy.stats import randint, uniform
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline
import joblib
from sklearn.impute import SimpleImputer
from sklearn.base import BaseEstimator, TransformerMixin
import shap 
from sklearn.inspection import permutation_importance
from sklearn.feature_selection import mutual_info_classif
import duckdb as db
from sklearn.decomposition import PCA
from sqlalchemy import create_engine, text
import plotly.express as px

In [3]:
df = pd.read_csv('D:\IT\Python\sklearn\hotel-booking-project\data\processed_hotel_booking.csv')

<>:1: SyntaxWarning: invalid escape sequence '\I'
<>:1: SyntaxWarning: invalid escape sequence '\I'
C:\Users\bben2\AppData\Local\Temp\ipykernel_16600\3920494749.py:1: SyntaxWarning: invalid escape sequence '\I'
  df = pd.read_csv('D:\IT\Python\sklearn\hotel-booking-project\data\processed_hotel_booking.csv')


In [4]:
df.head(3)

,hotel,is_canceled,lead_time,arrival_date_year,arrival_date_month,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,adults,...,customer_type,adr,required_car_parking_spaces,total_of_special_requests,has_company,has_agent,total_guests,total_nights,room_changed,previous_bookings
0,Resort Hotel,0,342,2015,July,27,1,0,0,2,...,Transient,0.0,0,0,0,0,2.0,0,0,0
1,Resort Hotel,0,737,2015,July,27,1,0,0,2,...,Transient,0.0,0,0,0,0,2.0,0,0,0
2,Resort Hotel,0,7,2015,July,27,1,0,1,1,...,Transient,75.0,0,0,0,0,1.0,1,1,0


In [5]:
X = df.drop('is_canceled', axis=1)
y = df['is_canceled']
X_train, X_test, y_train, y_test = train_test_split(X, y, stratify=y, test_size=0.2, random_state=42)

In [6]:
num_cols = X.select_dtypes(include = np.number).columns.tolist()
cat_cols = X.select_dtypes(exclude = np.number).columns.tolist()

In [7]:
remove_cols = ['is_repeated_guest', 'has_company', 'has_agent', 'room_changed']
for col in remove_cols:
    num_cols.remove(col)
    cat_cols.append(col)

In [8]:
preprocessor = ColumnTransformer([
    ('num', StandardScaler(), num_cols),
    ('cat', OneHotEncoder(handle_unknown='ignore'), cat_cols)])

In [9]:
pipe_xgb = Pipeline([
    ('prep', preprocessor),
    ('model', xgb.XGBClassifier())
])

In [10]:
pipe_xgb.fit(X_train, y_train)
preds = pipe_xgb.predict(X_test)
acc = accuracy_score(y_test, preds)
probs = pipe_xgb.predict_proba(X_test)[:,1]
roc = roc_auc_score(y_test, probs)
con_mat = confusion_matrix(y_test, preds)
cl = classification_report(y_test, preds)

In [11]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
grid_param = {
    'model__n_estimators': [100, 300, 500],
    'model__max_depth': [3, 5, 7],
    'model__learning_rate': [0.01, 0.03, 0.05],
    'model__subsample': [0.7, 0.9, 1.0],
    'model__colsample_bytree': [0.7, 0.9, 1.0]
}
grid = RandomizedSearchCV(
    pipe_xgb,
    grid_param,
    cv = skf,
    n_iter = 50,
    scoring = 'roc_auc',
    n_jobs = -1,
    random_state=42,
    verbose=1
)
grid.fit(X_train, y_train)
print(f"Best params {grid.best_params_}")
print(f"Best score {grid.best_score_}")

Fitting 5 folds for each of 50 candidates, totalling 250 fits
Best params {'model__subsample': 0.7, 'model__n_estimators': 500, 'model__max_depth': 7, 'model__learning_rate': 0.05, 'model__colsample_bytree': 1.0}
Best score 0.9488830842142196


In [13]:
best_model = grid.best_estimator_
best_preds = best_model.predict(X_test)
a = accuracy_score(y_test, best_preds)
best_probs = best_model.predict_proba(X_test)[:, 1]
r = roc_auc_score(y_test, best_probs)
cl = classification_report(y_test, best_preds)
f1 = f1_score(y_test, best_preds)


In [14]:
print(f'accuracy {a}')
print(f'roc_auc {r}')
print(f'classification report {cl}')

accuracy 0.8763715554066505
roc_auc 0.9506591960847997
classification report               precision    recall  f1-score   support

           0       0.89      0.92      0.90     15033
           1       0.86      0.80      0.83      8845

    accuracy                           0.88     23878
   macro avg       0.87      0.86      0.87     23878
weighted avg       0.88      0.88      0.88     23878



In [16]:
best_f1 = f1
preds_best = None
best_thereshold = None
theresholds = np.linspace(0.1, 0.7, 13)
for i in theresholds:
    temp_preds = (best_probs > i).astype(int)
    temp_f1 = f1_score(y_test, temp_preds)
    if temp_f1 > best_f1:
        best_f1 = temp_f1
        best_thereshold = i
        preds_best = temp_preds
    print(f'Thereshold {i}, f1 score = {temp_f1}')
print(f'Best thereshold {best_thereshold}, f1 score = {best_f1}')

Thereshold 0.1, f1 score = 0.7422539424761742
Thereshold 0.15, f1 score = 0.7730781384011195
Thereshold 0.2, f1 score = 0.7951567894811522
Thereshold 0.25, f1 score = 0.81463173504696
Thereshold 0.3, f1 score = 0.8270599154377642
Thereshold 0.35, f1 score = 0.8316250266922912
Thereshold 0.4, f1 score = 0.8357650167923801
Thereshold 0.44999999999999996, f1 score = 0.8352807714123653
Thereshold 0.5, f1 score = 0.827186512118019
Thereshold 0.5499999999999999, f1 score = 0.8196582744671859
Thereshold 0.6, f1 score = 0.8089957637677548
Thereshold 0.6499999999999999, f1 score = 0.7902476780185759
Thereshold 0.7, f1 score = 0.7659859390693003
Best thereshold 0.4, f1 score = 0.8357650167923801


In [19]:
print(classification_report(y_test, preds_best))

              precision    recall  f1-score   support

           0       0.91      0.89      0.90     15033
           1       0.81      0.86      0.84      8845

    accuracy                           0.88     23878
   macro avg       0.86      0.87      0.87     23878
weighted avg       0.88      0.88      0.88     23878

